# IBGE Municipality Population - Silver Transformation

## Setup

In [0]:
from pyspark.sql.functions import col, trim, split, lower, translate

environment = "dev"

catalog = f"ecommerce_{environment}"
source_table = f"{catalog}.bronze.ibge_municipality_population"
target_table = f"{catalog}.silver.ibge_municipality_population"

## Read Bronze data

In [0]:
bronze_df = spark.table(source_table)

## Review the data

In [0]:
bronze_df.printSchema()

root
 |-- ibge_municipality_id: string (nullable = true)
 |-- municipality_name: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- year: string (nullable = true)
 |-- population: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(bronze_df.limit(10))

ibge_municipality_id,municipality_name,source_file_path,source_file_modification_time,year,population,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
1100015,Alta Floresta D'Oeste (RO),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172653.json,2026-08-12T17:26:53.000Z,2016,25506,2026-08-12T17:34:04.683Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population
1100023,Ariquemes (RO),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172653.json,2026-08-12T17:26:53.000Z,2016,105896,2026-08-12T17:34:04.683Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population
1100031,Cabixi (RO),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172653.json,2026-08-12T17:26:53.000Z,2016,6289,2026-08-12T17:34:04.683Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population
1100049,Cacoal (RO),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172653.json,2026-08-12T17:26:53.000Z,2016,87877,2026-08-12T17:34:04.683Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population
1100056,Cerejeiras (RO),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172653.json,2026-08-12T17:26:53.000Z,2016,17959,2026-08-12T17:34:04.683Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population
1100064,Colorado do Oeste (RO),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172653.json,2026-08-12T17:26:53.000Z,2016,18639,2026-08-12T17:34:04.683Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population
1100072,Corumbiara (RO),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172653.json,2026-08-12T17:26:53.000Z,2016,8749,2026-08-12T17:34:04.683Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population
1100080,Costa Marques (RO),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172653.json,2026-08-12T17:26:53.000Z,2016,17031,2026-08-12T17:34:04.683Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population
1100098,Espigão D'Oeste (RO),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172653.json,2026-08-12T17:26:53.000Z,2016,32712,2026-08-12T17:34:04.683Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population
1100106,Guajará-Mirim (RO),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172653.json,2026-08-12T17:26:53.000Z,2016,47048,2026-08-12T17:34:04.683Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population


In [0]:
print("Number of rows:", bronze_df.count())
print("Number of columns:", len(bronze_df.columns))

Number of rows: 16713
Number of columns: 10


In [0]:
for column, dtype in bronze_df.dtypes[:2]+bronze_df.dtypes[4:6]:
    print(column)
    print("Null count:", bronze_df.filter(col(column).isNull()).count())
    print("Distinct count:", bronze_df.filter(col(column).isNotNull()).select(column).distinct().count())

    if dtype == "string":
        print("Extra whitespace row count:",
            (
                bronze_df.withColumn(f"{column}_trimmed", trim(col(column)))
                .filter(col(column) != col(f"{column}_trimmed"))
                .count()
            )
        )
    print("-"*20)

ibge_municipality_id
Null count: 0
Distinct count: 5571
Extra whitespace row count: 0
--------------------
municipality_name
Null count: 0
Distinct count: 5571
Extra whitespace row count: 0
--------------------
year
Null count: 0
Distinct count: 3
Extra whitespace row count: 0
--------------------
population
Null count: 0
Distinct count: 13156
Extra whitespace row count: 0
--------------------


In [0]:
bronze_df.select("ibge_municipality_id", "year").distinct().count()

16713

- ibge_municipality_id + year columns form a composite key for this table.
- There are no null values or extra whitespace in any column.

In [0]:
display(bronze_df.select('municipality_name').limit(10))

municipality_name
Alta Floresta D'Oeste (RO)
Ariquemes (RO)
Cabixi (RO)
Cacoal (RO)
Cerejeiras (RO)
Colorado do Oeste (RO)
Corumbiara (RO)
Costa Marques (RO)
Espigão D'Oeste (RO)
Guajará-Mirim (RO)


municipality name contains state code. These need to be separated.

## Transform to Silver

In [0]:
silver_df = bronze_df.withColumnRenamed("municipality_name", "municipality_name_raw")

In [0]:
silver_df = (
    silver_df
    .withColumn(
        "municipality_name",
        split(col("municipality_name_raw"), ' \\(')[0]
        )
    .withColumn(
        "state_code",
        split(split(col("municipality_name_raw"), ' \\(')[1], '\\)')[0]
    )
)

In [0]:
silver_df = silver_df.withColumn(
    "municipality_name",
    translate(
        lower(col("municipality_name")),
        "áàâãäéèêëíìîïóòôõöúùûüç-'",
        "aaaaaeeeeiiiiooooouuuuc  "
    )
)

In [0]:
silver_df = silver_df.withColumn("year", col("year").cast("int"))

silver_df = (
    silver_df
    .filter(
        col("population")
        .rlike("^[0-9]+$")
        )
    .withColumn(
        "population",
        col("population").cast("int"))
)

In [0]:
display(silver_df.select('municipality_name', "state_code", "year", "population").limit(20))

municipality_name,state_code,year,population
alta floresta d oeste,RO,2016,25506
ariquemes,RO,2016,105896
cabixi,RO,2016,6289
cacoal,RO,2016,87877
cerejeiras,RO,2016,17959
colorado do oeste,RO,2016,18639
corumbiara,RO,2016,8749
costa marques,RO,2016,17031
espigao d oeste,RO,2016,32712
guajara mirim,RO,2016,47048


## Write to Silver

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

## Review the result

In [0]:
silver_table_df = spark.table(target_table)

silver_table_df.printSchema()

root
 |-- ibge_municipality_id: string (nullable = true)
 |-- municipality_name_raw: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- year: integer (nullable = true)
 |-- population: integer (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)
 |-- municipality_name: string (nullable = true)
 |-- state_code: string (nullable = true)



In [0]:
display(silver_table_df.limit(10))

ibge_municipality_id,municipality_name_raw,source_file_path,source_file_modification_time,year,population,ingestion_timestamp,ingestion_run_id,source_system,source_dataset,municipality_name,state_code
1100015,Alta Floresta D'Oeste (RO),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172653.json,2026-08-12T17:26:53.000Z,2016,25506,2026-08-12T17:34:04.683Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population,alta floresta d oeste,RO
1100023,Ariquemes (RO),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172653.json,2026-08-12T17:26:53.000Z,2016,105896,2026-08-12T17:34:04.683Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population,ariquemes,RO
1100031,Cabixi (RO),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172653.json,2026-08-12T17:26:53.000Z,2016,6289,2026-08-12T17:34:04.683Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population,cabixi,RO
1100049,Cacoal (RO),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172653.json,2026-08-12T17:26:53.000Z,2016,87877,2026-08-12T17:34:04.683Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population,cacoal,RO
1100056,Cerejeiras (RO),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172653.json,2026-08-12T17:26:53.000Z,2016,17959,2026-08-12T17:34:04.683Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population,cerejeiras,RO
1100064,Colorado do Oeste (RO),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172653.json,2026-08-12T17:26:53.000Z,2016,18639,2026-08-12T17:34:04.683Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population,colorado do oeste,RO
1100072,Corumbiara (RO),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172653.json,2026-08-12T17:26:53.000Z,2016,8749,2026-08-12T17:34:04.683Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population,corumbiara,RO
1100080,Costa Marques (RO),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172653.json,2026-08-12T17:26:53.000Z,2016,17031,2026-08-12T17:34:04.683Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population,costa marques,RO
1100098,Espigão D'Oeste (RO),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172653.json,2026-08-12T17:26:53.000Z,2016,32712,2026-08-12T17:34:04.683Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population,espigao d oeste,RO
1100106,Guajará-Mirim (RO),dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipality_population/20260812/ibge_municipality_population_api_response_20260812_172653.json,2026-08-12T17:26:53.000Z,2016,47048,2026-08-12T17:34:04.683Z,9272ac9f-f400-41a2-aa20-307c2474a22b,ibge,municipality_population,guajara mirim,RO


In [0]:
print("Bronze row count:", bronze_df.count())
print("Silver row count:", silver_table_df.count())

Bronze row count: 16713
Silver row count: 16710


Silver contains 16,710 rows. Three Bronze rows for "boa esperanca do norte" are excluded because IBGE returned "..." instead of numeric values. The original source values are preserved in Bronze.